# B2S 02 - AndinaLog Flota

Conversión auditada de la fuente Bronze de flota a Silver y cuarentena. El archivo Bronze se lee sin modificar y todas las decisiones se centralizan en `CONFIG`.

In [2]:
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)


def find_root():
    candidates = []
    if os.getenv("ANDINALOG_ROOT"):
        candidates.append(Path(os.environ["ANDINALOG_ROOT"]))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "datos" / "bronze").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró el directorio datos/bronze")


ROOT = find_root()
EXECUTED_AT_UTC = datetime.now(timezone.utc).isoformat()
CONFIG = {
    "entidad": "camion",
    "granularidad": "una fila por camion_id normalizado",
    "clave": "camion_id",
    "rutas": {
        "bronze": "datos/bronze/andinalog_flota.csv",
        "notebook": "notebooks/bronze_silver/02_flota/B2S_02_AndinaLog_Flota.ipynb",
        "silver": "datos/silver/andinalog_flota_silver.csv",
        "quarantine": "datos/quarantine/andinalog_flota_quarantine.csv",
        "informe": "informes/bronze_silver/Informe_B2S_02_Flota.md",
    },
    "lectura": {"encoding": "utf-8", "dtype": "str", "keep_default_na": False},
    "columnas": {
        "texto": ["camion_id", "centro_distribucion_base", "tipo_camion"],
        "numericas": ["capacidad_kg"],
        "obligatorias": ["camion_id", "centro_distribucion_base", "capacidad_kg", "tipo_camion"],
    },
    "formatos": {
        "camion_id": r"^CAM-\d{2}$",
        "evidencia": "todos los identificadores observados usan dos dígitos; el patrón de tres dígitos del plan contradice la fuente",
    },
    "centros_validos": ["Cochabamba", "La Paz", "Santa Cruz", "Oruro", "Tarija"],
    "tipos_camion_validos": ["Seco", "Refrigerado"],
    "capacidad": {
        "unidad": "kg",
        "requiere_valor_positivo": True,
        "minimo_operacional": None,
        "motivo_sin_minimo": "no existe evidencia ni regla de dominio para inventar una capacidad mínima operacional",
    },
    "centinelas": [-999],
    "imputaciones": {
        "capacidad_kg": {
            "habilitada": False,
            "metodo": "",
            "motivo": "no hay ausencias y no existe una regla inequívoca aprobada",
        }
    },
    "politica_duplicados": {
        "metodo": "cuarentena_de_todas_las_ocurrencias",
        "motivo": "no elegir arbitrariamente una ocurrencia de una clave funcional duplicada",
    },
    "zonas_horarias": {
        "aplica_a_esta_fuente": False,
        "sin_zona": "America/La_Paz",
        "silver": "UTC",
    },
}
PATHS = {name: ROOT / relative for name, relative in CONFIG["rutas"].items()}
print("Raíz detectada:", ROOT)
print("Fuente:", CONFIG["rutas"]["bronze"])
print("Fecha de ejecución UTC:", EXECUTED_AT_UTC)


Raíz detectada: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Fuente: datos/bronze/andinalog_flota.csv
Fecha de ejecución UTC: 2026-09-25T02:48:38.278350+00:00


In [3]:
bronze = pd.read_csv(PATHS["bronze"], **CONFIG["lectura"])
normalized_ids = bronze["camion_id"].str.strip().str.upper()
capacidades_observadas = sorted(pd.to_numeric(bronze["capacidad_kg"], errors="coerce").dropna().unique().tolist())
perfil = {
    "filas": len(bronze),
    "columnas": bronze.columns.tolist(),
    "tipos_recibidos": bronze.dtypes.astype(str).to_dict(),
    "vacios": bronze.eq("").sum().to_dict(),
    "filas_clave_duplicada": int(normalized_ids.duplicated(keep=False).sum()),
    "claves_duplicadas": sorted(normalized_ids[normalized_ids.duplicated(keep=False)].unique().tolist()),
    "centros": bronze["centro_distribucion_base"].value_counts(dropna=False).to_dict(),
    "tipos_camion": bronze["tipo_camion"].value_counts(dropna=False).to_dict(),
    "capacidades_kg": capacidades_observadas,
    "capacidad_min_observada": min(capacidades_observadas),
    "capacidad_max_observada": max(capacidades_observadas),
}
print("Perfil Bronze")
for key, value in perfil.items():
    print(f"{key}: {value}")
print("Contradicciones del plan detectadas:")
print("- Duplicadas reales: CAM-01 y CAM-27; no solo CAM-01.")
print("- Formato real: CAM- seguido de dos dígitos; no tres.")


Perfil Bronze
filas: 32
columnas: ['camion_id', 'centro_distribucion_base', 'capacidad_kg', 'tipo_camion']
tipos_recibidos: {'camion_id': 'str', 'centro_distribucion_base': 'str', 'capacidad_kg': 'str', 'tipo_camion': 'str'}
vacios: {'camion_id': 0, 'centro_distribucion_base': 0, 'capacidad_kg': 0, 'tipo_camion': 0}
filas_clave_duplicada: 4
claves_duplicadas: ['CAM-01', 'CAM-27']
centros: {'Cochabamba': 14, 'La Paz': 6, 'Santa Cruz': 5, 'Oruro': 4, 'Tarija': 3}
tipos_camion: {'Refrigerado': 16, 'Seco': 16}
capacidades_kg: [2000, 5000, 8000, 12000]
capacidad_min_observada: 2000
capacidad_max_observada: 12000
Contradicciones del plan detectadas:
- Duplicadas reales: CAM-01 y CAM-27; no solo CAM-01.
- Formato real: CAM- seguido de dos dígitos; no tres.


In [4]:
def append_reason(df, mask, column, reason):
    df.loc[mask, column] = df.loc[mask, column].map(
        lambda current: reason if not current else f"{current} | {reason}"
    )
    return df


def estructurar(df):
    df = df.copy()
    df["_fila_bronze"] = range(2, len(df) + 2)
    for column in ["errores_bloqueantes", "motivos_transformacion", "motivos_imputacion"]:
        df[column] = ""
    for column in CONFIG["columnas"]["texto"] + CONFIG["columnas"]["numericas"]:
        df[f"{column}_original"] = df[column]
    return df


def normalizar_texto(df):
    df = df.copy()
    df["camion_id_tratado"] = df["camion_id"].str.strip().str.upper()
    df["centro_distribucion_base_tratado"] = df["centro_distribucion_base"].str.strip()
    df["tipo_camion_tratado"] = df["tipo_camion"].str.strip()
    for column in CONFIG["columnas"]["texto"]:
        flag = f"{column}_transformado"
        df[flag] = df[column].ne(df[f"{column}_tratado"])
        append_reason(df, df[flag], "motivos_transformacion", f"normalizacion_texto:{column}")
    return df


def convertir_capacidad(df):
    df = df.copy()
    source = "capacidad_kg"
    treated = "capacidad_kg_tratado"
    df[treated] = pd.to_numeric(df[source], errors="coerce")
    df["capacidad_kg_conversion_invalida"] = df[treated].isna() & df[source].ne("")
    append_reason(df, df["capacidad_kg_conversion_invalida"], "errores_bloqueantes", "conversion_invalida:capacidad_kg")
    df["capacidad_kg_centinela_detectado"] = df[treated].isin(CONFIG["centinelas"])
    df.loc[df["capacidad_kg_centinela_detectado"], treated] = pd.NA
    append_reason(df, df["capacidad_kg_centinela_detectado"], "errores_bloqueantes", "centinela:capacidad_kg")
    return df


def validar_catalogos_y_clave(df):
    df = df.copy()
    key = "camion_id_tratado"
    invalid_key = ~df[key].fillna("").str.match(CONFIG["formatos"]["camion_id"])
    duplicate = df[key].duplicated(keep=False) & df[key].notna()
    invalid_center = ~df["centro_distribucion_base_tratado"].isin(CONFIG["centros_validos"])
    invalid_type = ~df["tipo_camion_tratado"].isin(CONFIG["tipos_camion_validos"])
    append_reason(df, invalid_key, "errores_bloqueantes", "camion_id_invalido")
    append_reason(df, duplicate, "errores_bloqueantes", "camion_id_duplicado")
    append_reason(df, invalid_center, "errores_bloqueantes", "centro_distribucion_invalido")
    append_reason(df, invalid_type, "errores_bloqueantes", "tipo_camion_invalido")
    return df


def imputar(df):
    df = df.copy()
    df["capacidad_kg_imputada"] = False
    df["capacidad_kg_imputacion_metodo"] = ""
    return df


def validar_post_tratamiento(df):
    df = df.copy()
    for column in CONFIG["columnas"]["obligatorias"]:
        missing = df[f"{column}_tratado"].eq("") if column in CONFIG["columnas"]["texto"] else df[f"{column}_tratado"].isna()
        append_reason(df, missing, "errores_bloqueantes", f"requerido_ausente:{column}")
    if CONFIG["capacidad"]["requiere_valor_positivo"]:
        invalid_capacity = df["capacidad_kg_tratado"].notna() & df["capacidad_kg_tratado"].le(0)
        append_reason(df, invalid_capacity, "errores_bloqueantes", "capacidad_no_positiva")
    return df


def asignar_calidad(df):
    df = df.copy()
    transform_flags = [f"{column}_transformado" for column in CONFIG["columnas"]["texto"]]
    df["fue_transformada"] = df[transform_flags].any(axis=1)
    df["fue_imputada"] = df["capacidad_kg_imputada"]
    df["conteo_transformaciones"] = df[transform_flags].sum(axis=1).astype(int)
    df["conteo_imputaciones"] = df["fue_imputada"].astype(int)
    df["calidad_motivo"] = df["errores_bloqueantes"].mask(df["errores_bloqueantes"].eq(""), "sin_incidencias")
    df["calidad_estado"] = "valida"
    df.loc[df["fue_transformada"], "calidad_estado"] = "valida_con_transformacion"
    df.loc[df["fue_imputada"], "calidad_estado"] = "valida_con_imputacion"
    df.loc[df["fue_transformada"] & df["fue_imputada"], "calidad_estado"] = "valida_con_transformacion_e_imputacion"
    df.loc[df["errores_bloqueantes"].ne(""), "calidad_estado"] = "cuarentena"
    return df


work = (
    bronze.pipe(estructurar)
    .pipe(normalizar_texto)
    .pipe(convertir_capacidad)
    .pipe(validar_catalogos_y_clave)
    .pipe(imputar)
    .pipe(validar_post_tratamiento)
    .pipe(asignar_calidad)
)
silver = work.loc[work["calidad_estado"].ne("cuarentena")].copy()
quarantine = work.loc[work["calidad_estado"].eq("cuarentena")].copy()
for column in CONFIG["columnas"]["texto"] + CONFIG["columnas"]["numericas"]:
    silver[column] = silver[f"{column}_tratado"]
    quarantine[column] = quarantine[f"{column}_tratado"]
silver.to_csv(PATHS["silver"], index=False, encoding="utf-8")
quarantine.to_csv(PATHS["quarantine"], index=False, encoding="utf-8")
print({"bronze": len(bronze), "silver": len(silver), "quarantine": len(quarantine)})
print("Estados Silver:", silver["calidad_estado"].value_counts().to_dict())
print("Motivos cuarentena:", quarantine["errores_bloqueantes"].value_counts().to_dict())


{'bronze': 32, 'silver': 28, 'quarantine': 4}
Estados Silver: {'valida': 26, 'valida_con_transformacion': 2}
Motivos cuarentena: {'camion_id_duplicado': 4}


In [5]:
silver_file = pd.read_csv(PATHS["silver"])
quarantine_file = pd.read_csv(PATHS["quarantine"])
assert len(bronze) == len(silver_file) + len(quarantine_file)
assert silver_file["camion_id"].is_unique
assert silver_file["errores_bloqueantes"].fillna("").eq("").all()
assert set(silver_file["centro_distribucion_base"]).issubset(CONFIG["centros_validos"])
assert set(silver_file["tipo_camion"]).issubset(CONFIG["tipos_camion_validos"])
assert silver_file["capacidad_kg"].notna().all()
assert silver_file["capacidad_kg"].gt(0).all()
assert not silver_file["fue_imputada"].any()
audit_columns = {
    "camion_id_original", "camion_id_tratado", "camion_id_transformado",
    "capacidad_kg_original", "capacidad_kg_tratado", "capacidad_kg_conversion_invalida",
    "capacidad_kg_centinela_detectado", "capacidad_kg_imputada",
    "capacidad_kg_imputacion_metodo", "fue_transformada", "fue_imputada",
    "conteo_transformaciones", "conteo_imputaciones", "calidad_estado", "calidad_motivo",
}
assert audit_columns.issubset(silver_file.columns)

quality = silver_file["calidad_estado"].value_counts().to_dict()
reasons = quarantine_file["errores_bloqueantes"].value_counts().to_dict()
report = [
    "# Informe B2S 02 - AndinaLog Flota", "",
    "## Objetivo, entidad y granularidad",
    "Conversión auditada de la fuente Bronze de flota a Silver y cuarentena.",
    f"- Entidad: {CONFIG['entidad']}.",
    f"- Granularidad: {CONFIG['granularidad']}.",
    f"- Clave funcional: `{CONFIG['clave']}` normalizado.", "",
    "## Contrato de entrada",
    f"- Columnas obligatorias: {CONFIG['columnas']['obligatorias']}.",
    f"- Lectura Bronze: {CONFIG['lectura']}.",
    f"- Centros válidos: {CONFIG['centros_validos']}.",
    f"- Tipos de camión válidos: {CONFIG['tipos_camion_validos']}.", "",
    "## Perfil Bronze de esta ejecución",
    f"- Filas: {perfil['filas']}.",
    f"- Columnas: {perfil['columnas']}.",
    f"- Tipos recibidos: {perfil['tipos_recibidos']}.",
    f"- Valores vacíos: {perfil['vacios']}.",
    f"- Claves duplicadas normalizadas: {perfil['claves_duplicadas']} ({perfil['filas_clave_duplicada']} filas).",
    f"- Centros observados: {perfil['centros']}.",
    f"- Tipos observados: {perfil['tipos_camion']}.",
    f"- Capacidades observadas: {perfil['capacidades_kg']} kg; rango observado {perfil['capacidad_min_observada']} a {perfil['capacidad_max_observada']} kg.", "",
    "## Reglas y decisiones",
    f"- Formato de identificador aplicado: `{CONFIG['formatos']['camion_id']}`; evidencia: {CONFIG['formatos']['evidencia']}.",
    "- Se normalizan espacios y mayúsculas de identificadores; las correcciones inequívocas pueden llegar a Silver con auditoría.",
    "- Centro y tipo se validan contra los catálogos aprobados.",
    f"- No se fija capacidad mínima operacional: {CONFIG['capacidad']['motivo_sin_minimo']}. Solo se exige capacidad numérica positiva por significado físico.",
    f"- Imputación de capacidad deshabilitada: {CONFIG['imputaciones']['capacidad_kg']['motivo']}.",
    "- No hay fechas en la fuente; la regla America/La_Paz a UTC no aplica.", "",
    "## Contradicciones detectadas respecto del plan",
    "- La fuente contiene dos claves duplicadas, `CAM-01` y `CAM-27`; se aplica la política aprobada a las cuatro ocurrencias.",
    "- La fuente usa identificadores de dos dígitos; se detuvo la regla incompatible de tres dígitos y se validó el patrón observado, sin inventar otra codificación.", "",
    "## Enrutamiento y controles",
    f"- Política de duplicados: {CONFIG['politica_duplicados']['metodo']} ({CONFIG['politica_duplicados']['motivo']}).",
    f"- Silver: {len(silver_file)} filas; estados: {quality}.",
    f"- Filas imputadas: {int(silver_file['fue_imputada'].sum())}.",
    f"- Cuarentena: {len(quarantine_file)} filas; motivos: {reasons}.",
    f"- Conciliación persistida: Bronze {len(bronze)} = Silver {len(silver_file)} + cuarentena {len(quarantine_file)}.",
    "- Silver tiene clave única y no contiene errores bloqueantes.", "",
    "## Limitaciones",
    "- La fuente no contiene una regla de precedencia para resolver duplicados.",
    "- No existe evidencia para establecer una capacidad mínima operacional distinta de la validación física de positividad.", "",
    "## Archivos generados",
    f"- `{CONFIG['rutas']['notebook']}`",
    f"- `{CONFIG['rutas']['silver']}`",
    f"- `{CONFIG['rutas']['quarantine']}`",
    f"- `{CONFIG['rutas']['informe']}`", "",
    "## Reproducibilidad",
    f"- Fecha de ejecución UTC: {EXECUTED_AT_UTC}.",
    f"- Python: {platform.python_version()}.",
    f"- pandas: {pd.__version__}.",
    f"- Ruta Bronze relativa: `{CONFIG['rutas']['bronze']}`.",
    f"- Ruta Silver relativa: `{CONFIG['rutas']['silver']}`.",
    f"- Ruta cuarentena relativa: `{CONFIG['rutas']['quarantine']}`.",
    f"- Zona de fechas sin zona: {CONFIG['zonas_horarias']['sin_zona']}; destino Silver: {CONFIG['zonas_horarias']['silver']}; no aplica por ausencia de fechas.",
    f"- Conteos: Bronze {len(bronze)}, Silver {len(silver_file)}, cuarentena {len(quarantine_file)}.",
    "- Bronze se lee con `dtype=str` y no se modifica; los controles finales se ejecutan sobre los CSV persistidos.",
]
PATHS["informe"].write_text("\n".join(report) + "\n", encoding="utf-8")
print("Controles persistidos: OK")
print(pd.DataFrame({"bronze": [len(bronze)], "silver": [len(silver_file)], "quarantine": [len(quarantine_file)]}))
print("Informe generado:", PATHS["informe"])


Controles persistidos: OK
   bronze  silver  quarantine
0      32      28           4
Informe generado: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\bronze_silver\Informe_B2S_02_Flota.md
